# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
# Print summary
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List record sets and their fields.

def print_record_sets(ds):
    if not hasattr(ds.metadata, 'record_sets') or not ds.metadata.record_sets:
        print('No record sets available in metadata.')
        return []
    print('Available Record Sets:')
    record_set_ids = []
    for rs in ds.metadata.record_sets:
        print(f"- Record Set ID: {rs['@id']}")
        name = rs.get('name', '')
        print(f"  Name: {name}")
        print('  Fields:')
        if 'fields' in rs:
            for field in rs['fields']:
                if isinstance(field, dict):
                    fid = field.get('@id', '')
                    fname = field.get('name', '')
                    print(f"    - {fid} ({fname})")
                else:
                    print(f"    - {field}")
        else:
            print('    [No fields]')
        record_set_ids.append(rs['@id'])
    return record_set_ids

record_set_ids = print_record_sets(dataset)

# If the metadata does not expose record_sets, enumerate from files in the Dataset
if not record_set_ids:
    # Fallback: Attempt to use Dataset's record sets as available
    # Use dataset._record_sets for advanced manual access
    print('\nEnumerating record_set @id values from dataset:')
    rs_ids = []
    for rset in dataset._record_sets:
        print(f"- @id: {rset['@id']} | name: {rset.get('name', '')}")
        if 'fields' in rset:
            print('  Fields:')
            for f in rset['fields']:
                fid = f.get('@id') if isinstance(f, dict) else f
                fname = f.get('name', '') if isinstance(f, dict) else ''
                print(f"    - {fid} ({fname})")
        rs_ids.append(rset['@id'])
    record_set_ids = rs_ids

print(f"\nRecord Sets found: {record_set_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract record set data as DataFrames
import itertools

if not record_set_ids:
    raise ValueError("No record sets found in the dataset.")

dataframes = {}

# Show the first N records in each record set
for record_set_id in record_set_ids:
    print(f'Loading records from record set: {record_set_id}')
    records_iterator = dataset.records(record_set=record_set_id)
    first5 = list(itertools.islice(records_iterator, 5))
    print(f"First 5 records from '{record_set_id}':")
    for rec in first5:
        print(rec)

    # Reload all records for this record set into DataFrame
    df = pd.DataFrame(list(dataset.records(record_set=record_set_id)))
    print(f"Columns for {record_set_id}: {df.columns.tolist()}")
    dataframes[record_set_id] = df
    print(df.head(2))

# For further analysis, pick the first record set
main_record_set_id = record_set_ids[0]
main_df = dataframes[main_record_set_id]
print(f"\nUsing record set: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Suggest a numeric field for demonstration, e.g., 'cr:MW_kDa' or 'cr:AbundanceSample1'

print("Available columns:", main_df.columns.tolist())

# Try to pick a numeric field
numeric_candidates = [col for col in main_df.columns if any(k in col.lower() for k in ['abundance', 'mw', 'kda', 'coverage', 'peptide', 'score', 'count', 'number', 'intensity'])]
print(f"Possible numeric fields: {numeric_candidates}")

if len(numeric_candidates) > 0:
    numeric_field = numeric_candidates[0]
else:
    numeric_field = main_df.columns[0]

print(f"Using numeric field for EDA: {numeric_field}")

# Convert to float if not already
main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors='coerce')

threshold = main_df[numeric_field].mean() if main_df[numeric_field].dtype.kind in 'fiu' else 10

# Filter records
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to group by a non-numeric field
category_candidates = [col for col in main_df.columns if not any(k in col.lower() for k in ['abundance', 'mw', 'kda', 'coverage', 'peptide', 'score', 'count', 'number', 'intensity'])]
group_field = None
for cat in category_candidates:
    if main_df[cat].nunique() > 1:
        group_field = cat
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped filtered data by '{group_field}':")
    print(grouped_df.head())
else:
    print("No suitable group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualize numeric data distribution and relationships
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
plt.xlabel(numeric_field)
plt.title(f'Histogram of {numeric_field}')
plt.show()

if group_field:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=group_field, y=numeric_field, data=main_df)
    plt.title(f'{numeric_field} by {group_field}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we loaded the `Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells` dataset using a Croissant schema and the `mlcroissant` library.
* We explored available record sets and their fields by their `@id`, loaded table data into DataFrames, and carried out basic data extraction.
* We performed simple filtering, normalization, and grouping operations for numeric fields, and visualized distributions for exploratory data analysis.
* For more advanced use, you can now apply custom data cleaning, complex statistics, and scientific modeling to the extracted data using standard Python data science tools.